## Introduction

This in-class project asks you to build a **Mini Data Profiler** — a utility that takes a list of numbers and produces a rich statistical summary, a text histogram, a normalized view of the data, and a streaming stats trace.

By the end of this project you should be able to:

* Use **OOP** to wrap a full utility into a clean, reusable class
* Apply **NumPy** array math, broadcasting, and aggregate operations
* Write and apply a **decorator** to measure execution time
* Use a **generator** to stream statistics one value at a time

Please make sure to run <span style="color: red;">all cells</span> in order. Hints are provided for every step — read them carefully before writing any code.


---
## What You Are Building

You will build a `DataProfiler` class. When given a list of numbers, it will:

1. Print a **statistics block** — count, mean, median, std, min, max
2. Print a **text histogram** — a simple bar chart using `█` characters
3. Return a **normalized array** — data scaled to `[0, 1]` using NumPy broadcasting
4. Compute **cosine similarity** between two arrays (if two datasets are given)
5. Run a **streaming stats generator** — yield `(mean, std)` after each new value
6. Wrap the main `profile()` method in a **`@timer` decorator** that prints elapsed time

A complete run should look like this:

```
=== Data Profile ===
Count  : 8
Mean   : 45.75
Median : 43.0
Std    : 22.3
Min    : 12    Max : 89

Histogram (4 bins):
[12 – 34] : ████   (2)
[34 – 56] : █████  (3)
[56 – 78] : ██     (1)
[78 – 89] : ██     (2)

Normalized : [0.   0.42 0.13 0.71 0.28 1.   0.56 0.38]

Streaming stats (first 5 values):
  n=1 → mean=23.00, std=0.00
  n=2 → mean=34.00, std=11.00
  ...

[profile() completed in 0.0003s]
```


---
## Step 1 — Imports

Import everything you will need up front.

> **Hint:** You need `numpy` for all array math, and `time` for the decorator.


In [ ]:
import numpy as np
import time

# Nothing to change here — just run this cell


---
## Step 2 — The `@timer` Decorator

Before building the class, write a standalone decorator called `timer`.

> **Hints:**
> - A decorator is a function that takes a function `func` and returns a `wrapper` function.
> - Inside `wrapper`, record the time before and after calling `func`, then print the difference.
> - Use `time.time()` to get the current time in seconds.
> - The wrapper must accept `*args` and `**kwargs` so it works on any function.
> - Print the elapsed time formatted to 4 decimal places, e.g. `[profile() completed in 0.0003s]`


In [ ]:
def timer(func):
    ### Code here
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"[{func.__name__}() completed in {elapsed:.4f}s]")
        return result
    return wrapper


# Quick test — when you run this, it should print something like:
# [slow_add() completed in 1.0001s]

@timer
def slow_add(a, b):
    time.sleep(1)
    return a + b

result = slow_add(3, 4)
print("Result:", result)




[slow_add() completed in 1.0001s]
Result: 7


---
## Step 3 — The `DataProfiler` Class Skeleton

Create the class with its `__init__` method.

> **Hints:**
> - Store the input data as a **NumPy array** using `np.array(data)` — not a plain list.
> - Store it as `self.data`.
> - The constructor should accept one argument: `data` (a list or array of numbers).


In [ ]:
class DataProfiler:

    def __init__(self, data):
        ### Code here — store data as a NumPy array
        self.data = np.array(data)

# Quick test
dp = DataProfiler([23, 45, 12, 67, 34, 89, 55, 41])
print(type(dp.data))   # should print: <class 'numpy.ndarray'>
print(dp.data)         # should print: [23 45 12 67 34 89 55 41]


<class 'numpy.ndarray'>
[23 45 12 67 34 89 55 41]


---
## Step 4 — Statistics Block

Add a method `stats(self)` that returns a formatted string with count, mean, median, std, min, and max.

> **Hints:**
> - Use `len(self.data)` for count.
> - Use `np.mean()`, `np.median()`, `np.std()`, `np.min()`, `np.max()` — all accept a NumPy array directly.
> - Round everything to 2 decimal places using Python's `round()` or f-string `:.2f` formatting.
> - Return a multi-line string using triple quotes or `\n` joins.


In [ ]:
class DataProfiler:

    def __init__(self, data):
        self.data = np.array(data)

    def stats(self):
        ### Code here
        return (
            "=== Statistics ===\n"
            f"Count  : {len(self.data)}\n"
            f"Mean   : {np.mean(self.data):.2f}\n"
            f"Median : {np.median(self.data):.1f}\n"
            f"Std    : {np.std(self.data):.1f}\n"
            f"Min    : {np.min(self.data):<5} Max : {np.max(self.data)}"
        )


dp = DataProfiler([23, 45, 12, 67, 34, 89, 55, 41])
print(dp.stats())
# Expected output:
# === Statistics ===
# Count  : 8
# Mean   : 45.75
# Median : 43.0
# Std    : 22.3
# Min    : 12    Max : 89


=== Statistics ===
Count  : 8
Mean   : 45.75
Median : 43.0
Std    : 23.0
Min    : 12    Max : 89


---
## Step 5 — Text Histogram

Add a method `histogram(self, bins=4)` that prints a text bar chart.

> **Hints:**
> - Use `np.histogram(self.data, bins=bins)` — it returns two arrays: `counts` and `edges`.
>   - `counts[i]` is how many values fall in bin `i`.
>   - `edges[i]` and `edges[i+1]` are the left and right boundaries of bin `i`.
> - Build each bar with string multiplication: `"█" * count` gives a bar of that length.
> - Loop over `zip(counts, edges)` — but remember `edges` has one extra value (the right edge of the last bin).
> - Format each line like: `[12 – 34] : ████   (2)`


In [ ]:
class DataProfiler:

    def __init__(self, data):
        self.data = np.array(data)

    def stats(self):
        # (copy your working stats method here)
       return (
            "=== Statistics ===\n"
            f"Count  : {len(self.data)}\n"
            f"Mean   : {np.mean(self.data):.2f}\n"
            f"Median : {np.median(self.data):.1f}\n"
            f"Std    : {np.std(self.data):.1f}\n"
            f"Min    : {np.min(self.data):<5} Max : {np.max(self.data)}"
        )

    def histogram(self, bins=4):
        ### Code here
        counts, edges = np.histogram(self.data, bins=bins)
        print(f"Histogram ({bins} bins):")
        for i, count in enumerate(counts):
            left, right = edges[i], edges[i + 1]
            bar = "█" * count
            print(f"[{left:.0f} – {right:.0f}] : {bar:<6} ({count})")


dp = DataProfiler([23, 45, 12, 67, 34, 89, 55, 41])
dp.histogram()
# Expected output (approximate — depends on data):
# Histogram (4 bins):
# [12 – 34] : ████   (2)
# [34 – 56] : █████  (3)
# [56 – 78] : ██     (1)
# [78 – 89] : ██     (2)


Histogram (4 bins):
[12 – 31] : ██     (2)
[31 – 50] : ███    (3)
[50 – 70] : ██     (2)
[70 – 89] : █      (1)


---
## Step 6 — Normalization

Add a method `normalize(self)` that returns the data scaled to the range `[0, 1]`.

> **Hints:**
> - The formula is: `(x - min) / (max - min)` applied to every element.
> - With NumPy you do NOT need a loop — just write the formula once using `self.data`, `self.data.min()`, and `self.data.max()`.
> - This is called **min-max normalization** and is used in almost every ML preprocessing pipeline.
> - Return the normalized array (don't modify `self.data`).


In [ ]:
class DataProfiler:

    def __init__(self, data):
        self.data = np.array(data)

    def stats(self):
         # copy your working version here
        return (
            "=== Statistics ===\n"
            f"Count  : {len(self.data)}\n"
            f"Mean   : {np.mean(self.data):.2f}\n"
            f"Median : {np.median(self.data):.1f}\n"
            f"Std    : {np.std(self.data):.1f}\n"
            f"Min    : {np.min(self.data):<5} Max : {np.max(self.data)}"
        )
    def histogram(self, bins=4):
        counts, edges = np.histogram(self.data, bins=bins)
        print(f"Histogram ({bins} bins):")
        for i, count in enumerate(counts):
            left, right = edges[i], edges[i + 1]
            bar = "█" * count
            print(f"[{left:.0f} – {right:.0f}] : {bar:<6} ({count})")
    def normalize(self):
        ### Code here
     return (self.data - self.data.min()) / (self.data.max() - self.data.min())


dp = DataProfiler([23, 45, 12, 67, 34, 89, 55, 41])
print(dp.normalize().round(2))
# Expected: [0.26 0.43 0.   0.71 0.28 1.   0.56 0.38]


[0.14 0.43 0.   0.71 0.29 1.   0.56 0.38]


---
## Step 7 — Streaming Stats Generator

Add a method `stream_stats(self)` that is a **generator**. It should yield a tuple `(mean, std)` after each new value is added from `self.data` — one yield per value.

> **Hints:**
> - Use a loop: `for i in range(1, len(self.data) + 1):`
> - Inside the loop, take a slice: `window = self.data[:i]`
> - Compute `np.mean(window)` and `np.std(window)`
> - Use `yield` (not `return`) to produce the tuple.
> - The caller will use a `for mean, std in dp.stream_stats():` loop to consume it.


In [ ]:
import numpy as np
class DataProfiler:

    def __init__(self, data):
        self.data = np.array(data)

    def stats(self):
         # copy your working version here
        return (
            "=== Statistics ===\n"
            f"Count  : {len(self.data)}\n"
            f"Mean   : {np.mean(self.data):.2f}\n"
            f"Median : {np.median(self.data):.1f}\n"
            f"Std    : {np.std(self.data):.1f}\n"
            f"Min    : {np.min(self.data):<5} Max : {np.max(self.data)}"
        )
    def histogram(self, bins=4):
     counts, edges = np.histogram(self.data, bins=bins)
     print(f"Histogram ({bins} bins):")
     for count, left, right in zip(counts, edges[:-1], edges[1:]):
        bar = "█" * count
        print(f"[{left:.0f} – {right:.0f}] : {bar} ({count})")

# edges[:-1] means "all of edges except the last one" → [12, 31.25, 50.5, 69.75] (the left boundary of each bin)
# edges[1:] means "all of edges except the first one" → [31.25, 50.5, 69.75, 89] (the right boundary of each bin)

    def normalize(self):
        ### Code here
     return (self.data - self.data.min()) / (self.data.max() - self.data.min())
    def stream_stats(self):
        ### Code here — remember to use yield, not return
          for i in range(1, len(self.data) + 1):
            window = self.data[:i]
            yield np.mean(window), np.std(window)


dp = DataProfiler([23, 45, 12, 67, 34, 89, 55, 41])
for i, (mean, std) in enumerate(dp.stream_stats(), 1):
    print(f"  n={i} → mean={mean:.2f}, std={std:.2f}")


          # counts, edges = np.histogram(self.data, bins=bins)
        # print(f"Histogram ({bins} bins):")
        # for i, count in enumerate(counts):
        #     left, right = edges[i], edges[i + 1]
        #     bar = "█" * count
        #     print(f"[{left:.0f} – {right:.0f}] : {bar:<6} ({count})")


  n=1 → mean=23.00, std=0.00
  n=2 → mean=34.00, std=11.00
  n=3 → mean=26.67, std=13.72
  n=4 → mean=36.75, std=21.12
  n=5 → mean=36.20, std=18.93
  n=6 → mean=45.00, std=26.19
  n=7 → mean=46.43, std=24.49
  n=8 → mean=45.75, std=22.98


---
## Step 8 — Putting It All Together

Now build the **final, complete `DataProfiler` class** with all methods together, and add:

- A `@timer` decorator applied to a `full_report(self)` method
- `full_report()` should call `stats()`, `histogram()`, `normalize()`, and print the first 5 streaming values
- A `__str__` that returns `DataProfiler | n=8 | mean=45.75`
- A `__len__` that returns the number of data points

> **Hints:**
> - Apply your decorator by putting `@timer` on the line directly above `def full_report(self):`.
> - For the streaming preview, use `enumerate` and `break` after 5 values.
> - `__len__` should return `len(self.data)`.


In [1]:
import numpy as np
import time


def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"[{func.__name__}() completed in {elapsed:.4f}s]")
        return result
    return wrapper


class DataProfiler:

    def __init__(self, data):
        self.data = np.array(data, dtype=float)

    def __str__(self):
        return f"DataProfiler | n={len(self.data)} | mean={np.mean(self.data):.2f}"

    def __len__(self):
        return len(self.data)

    def stats(self):
        return (
            "=== Statistics ===\n"
            f"Count  : {len(self.data)}\n"
            f"Mean   : {np.mean(self.data):.2f}\n"
            f"Median : {np.median(self.data):.1f}\n"
            f"Std    : {np.std(self.data):.1f}\n"
            f"Min    : {np.min(self.data):<5} Max : {np.max(self.data)}"
        )

    def histogram(self, bins=4):
        counts, edges = np.histogram(self.data, bins=bins)
        print(f"Histogram ({bins} bins):")
        for i, count in enumerate(counts):
            left, right = edges[i], edges[i + 1]
            bar = "█" * count
            print(f"[{left:.0f} – {right:.0f}] : {bar:<6} ({count})")

    def normalize(self):
        return (self.data - self.data.min()) / (self.data.max() - self.data.min())

    def stream_stats(self):
        for i in range(1, len(self.data) + 1):
            window = self.data[:i]
            yield np.mean(window), np.std(window)

    @timer
    def full_report(self):
        print(self.stats())
        print()
        self.histogram()
        print()
        print("Normalized:", self.normalize().round(2))
        print()
        print("Streaming stats (first 5):")
        for i, (mean, std) in enumerate(self.stream_stats(), 1):
            print(f"  n={i} → mean={mean:.2f}, std={std:.2f}")
            if i == 5:
                break


# ── Final test ──
dp = DataProfiler([23, 45, 12, 67, 34, 89, 55, 41])
print(dp)
print("Length:", len(dp))
print()
dp.full_report()



DataProfiler | n=8 | mean=45.75
Length: 8

=== Statistics ===
Count  : 8
Mean   : 45.75
Median : 43.0
Std    : 23.0
Min    : 12.0  Max : 89.0

Histogram (4 bins):
[12 – 31] : ██     (2)
[31 – 50] : ███    (3)
[50 – 70] : ██     (2)
[70 – 89] : █      (1)

Normalized: [0.14 0.43 0.   0.71 0.29 1.   0.56 0.38]

Streaming stats (first 5):
  n=1 → mean=23.00, std=0.00
  n=2 → mean=34.00, std=11.00
  n=3 → mean=26.67, std=13.72
  n=4 → mean=36.75, std=21.12
  n=5 → mean=36.20, std=18.93
[full_report() completed in 0.0063s]


In [4]:
################   FULL CODE     #####################
import numpy as np
import time


def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"[{func.__name__}() completed in {elapsed:.4f}s]")
        return result
    return wrapper


class DataProfiler:

    def __init__(self, data):
        self.data = np.array(data, dtype=float)   # force float dtype

    def __str__(self):
        return f"DataProfiler | n={len(self.data)} | mean={np.mean(self.data):.2f}"

    def __len__(self):
        return len(self.data)

    def stats(self):
        return (
            "=== Statistics ===\n"
            f"Count  : {len(self.data)}\n"
            f"Mean   : {np.mean(self.data):.2f}\n"
            f"Median : {np.median(self.data):.1f}\n"
            f"Std    : {np.std(self.data):.1f}\n"
            f"Min    : {np.min(self.data):<5} Max : {np.max(self.data)}"
        )

    def histogram(self, bins=4):
        counts, edges = np.histogram(self.data, bins=bins)
        print(f"Histogram ({bins} bins):")
        for i, count in enumerate(counts):
            left, right = edges[i], edges[i + 1]
            bar = "█" * count
            print(f"[{left:.0f} – {right:.0f}] : {bar:<6} ({count})")

    def normalize(self):
        return (self.data - self.data.min()) / (self.data.max() - self.data.min())

    def stream_stats(self):
        for i in range(1, len(self.data) + 1):
            window = self.data[:i]
            yield np.mean(window), np.std(window)

    @timer
    def full_report(self):
        print(self.stats())
        print()
        self.histogram()
        print()
        print("Normalized:", self.normalize().round(2))
        print()
        print("Streaming stats (first 5):")
        for i, (mean, std) in enumerate(self.stream_stats(), 1):
            print(f"  n={i} → mean={mean:.2f}, std={std:.2f}")
            if i == 5:
                break

    @staticmethod
    def cosine_similarity(a, b):
        a = np.asarray(a, dtype=float)
        b = np.asarray(b, dtype=float)
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# ── Final test ──
dp = DataProfiler([23, 45, 12, 67, 34, 89, 55, 41])
print(dp)
print("Length:", len(dp))
print()
dp.full_report()

# ── Bonus test ──
a = np.array([23, 45, 12, 67], dtype=float)
b = np.array([34, 89, 55, 41], dtype=float)
print("\nCosine similarity:", DataProfiler.cosine_similarity(a, b))



	# If all inputs are Python ints, self.data becomes an int array. Then normalize() still works (NumPy promotes to float on division),
  #  but self.data.min() / self.data.max() style math and any future integer operations can silently truncate.
  #  Forcing dtype=float is safer and makes intent explicit.

  # np.std by default computes the population standard deviation (ddof=0). If program expects the sample std, pass ddof=1:
 # np.std(self.data, ddof=1)   # sample std → 24.56 for this data




DataProfiler | n=8 | mean=45.75
Length: 8

=== Statistics ===
Count  : 8
Mean   : 45.75
Median : 43.0
Std    : 23.0
Min    : 12.0  Max : 89.0

Histogram (4 bins):
[12 – 31] : ██     (2)
[31 – 50] : ███    (3)
[50 – 70] : ██     (2)
[70 – 89] : █      (1)

Normalized: [0.14 0.43 0.   0.71 0.29 1.   0.56 0.38]

Streaming stats (first 5):
  n=1 → mean=23.00, std=0.00
  n=2 → mean=34.00, std=11.00
  n=3 → mean=26.67, std=13.72
  n=4 → mean=36.75, std=21.12
  n=5 → mean=36.20, std=18.93
[full_report() completed in 0.0020s]

Cosine similarity: 0.8232851068049467


---
## Bonus — Cosine Similarity

If you finish early, add a `@staticmethod` called `cosine_similarity(a, b)` that computes the cosine similarity between two 1D NumPy arrays.

The formula is:  `dot(a, b) / (||a|| × ||b||)`

> **Hints:**
> - Use `np.dot(a, b)` for the dot product.
> - Use `np.linalg.norm(a)` for the vector magnitude `||a||`.
> - A result of `1.0` means identical direction; `0.0` means completely unrelated.
> - This is used in NLP to compare text embeddings — one of the most common operations in AI engineering.


In [3]:
# Add cosine_similarity as a @staticmethod inside DataProfiler, then test it here:

a = np.array([23, 45, 12, 67], dtype=float)
b = np.array([34, 89, 55, 41], dtype=float)

print("Cosine similarity:", DataProfiler.cosine_similarity(a, b))
# Should be a value between 0 and 1


Cosine similarity: 0.8232851068049467
